# Phase 10 — La chaîne de traitement du Bureau

## Objectifs

- Construire une chaîne complète de préparation et de prédiction.
- Effectuer le découpage temporel avant tout apprentissage statistique.
- Apprendre les médianes et le vocabulaire uniquement sur le train.
- Vérifier la proportion de canulars dans le train et dans le test.
- Faire traverser toute la chaîne à un relevé inventé et obtenir une prédiction.


## 1. Imports des bibliothèques

In [ ]:
from pathlib import Path
import csv
import re

import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer


## 2. Chemins, colonnes et paramètres

In [ ]:
DATA_PATH = Path("../data/releves_klaxo3.csv")
OUTPUT_DIR = Path("../outputs")
PHASE10_DIR = OUTPUT_DIR / "phase_10_pipeline_complet"
PHASE10_DIR.mkdir(parents=True, exist_ok=True)

COLUMNS = [
    "datetime", "city", "state", "country", "shape",
    "duration_seconds", "duration_hours_min", "comments",
    "date_posted", "latitude", "longitude",
]

MOTS_CLES_CANULAR = [
    "hoax", "fake", "prank", "joke",
    "not real", "made up", "fraud",
]

TEST_SIZE = 0.20
RANDOM_STATE = 42


## 3. Chargement robuste des données

In [ ]:
lignes_valides = []
lignes_problemes = []

with open(
    DATA_PATH,
    "r",
    encoding="utf-8",
    errors="replace",
    newline="",
) as f:
    reader = csv.reader(f)

    for numero_ligne, row in enumerate(reader, start=1):
        if len(row) == len(COLUMNS):
            lignes_valides.append(row)
        else:
            lignes_problemes.append(
                {
                    "numero_ligne": numero_ligne,
                    "nb_champs": len(row),
                    "contenu": row,
                }
            )

df = pd.DataFrame(lignes_valides, columns=COLUMNS)

print(f"Lignes chargées : {len(df)}")
print(f"Lignes isolées : {len(lignes_problemes)}")


## 4. Conversions minimales et création de la cible

Les conversions techniques ne calculent pas de statistique globale. Les médianes et le vocabulaire seront appris ultérieurement dans le pipeline, uniquement sur le train.

In [ ]:
for col in ["duration_seconds", "latitude", "longitude"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

for col in ["datetime", "date_posted"]:
    df[col] = pd.to_datetime(df[col], errors="coerce")

for col in [
    "city", "state", "country", "shape",
    "duration_hours_min", "comments",
]:
    df[col] = (
        df[col]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
    )

pattern_canular = "|".join(
    re.escape(mot)
    for mot in MOTS_CLES_CANULAR
)

df["comments_clean"] = (
    df["comments"]
    .fillna("")
    .astype(str)
    .str.lower()
)

df["is_hoax"] = (
    df["comments_clean"]
    .str.contains(pattern_canular, regex=True, na=False)
    .astype(int)
)


## 5. Découpage temporel avant les calculs appris

In [ ]:
df_temporel = df.loc[df["datetime"].notna()].copy()
df_temporel = df_temporel.sort_values("datetime").copy()

position_coupure = int(len(df_temporel) * (1 - TEST_SIZE))
date_coupure = df_temporel.iloc[position_coupure]["datetime"]

df_train = df_temporel.loc[
    df_temporel["datetime"] < date_coupure
] .copy()

df_test = df_temporel.loc[
    df_temporel["datetime"] >= date_coupure
] .copy()

assert df_train["datetime"].max() < df_test["datetime"].min()

print(f"Date de coupure : {date_coupure}")
print(f"Train : {len(df_train)} relevés")
print(f"Test : {len(df_test)} relevés")


## 6. Proportion de canulars dans chaque jeu

In [ ]:
resume_repartition = pd.DataFrame(
    [
        {
            "jeu": "train",
            "nombre_releves": len(df_train),
            "nombre_canulars": int(df_train["is_hoax"].sum()),
            "proportion_canulars": df_train["is_hoax"].mean(),
            "date_min": df_train["datetime"].min(),
            "date_max": df_train["datetime"].max(),
        },
        {
            "jeu": "test",
            "nombre_releves": len(df_test),
            "nombre_canulars": int(df_test["is_hoax"].sum()),
            "proportion_canulars": df_test["is_hoax"].mean(),
            "date_min": df_test["datetime"].min(),
            "date_max": df_test["datetime"].max(),
        },
    ]
)

resume_repartition


## 7. Construction des données brutes d'entrée

Les entrées fournies au pipeline sont volontairement brutes. Aucun vocabulaire, aucune médiane et aucun encodage appris ne sont calculés avant le découpage.

In [ ]:
FEATURES_BRUTES = [
    "datetime",
    "city",
    "state",
    "country",
    "shape",
    "duration_seconds",
    "latitude",
    "longitude",
]

X_train = df_train[FEATURES_BRUTES].copy()
X_test = df_test[FEATURES_BRUTES].copy()

y_train = df_train["is_hoax"].copy()
y_test = df_test["is_hoax"].copy()


## 8. Fonction de préparation d'un relevé

Cette fonction crée uniquement des caractéristiques provenant de chaque ligne : conversions, composantes de date et texte combiné. Elle ne calcule pas de moyenne, de médiane ou de vocabulaire global.

In [ ]:
def preparer_releves(dataframe):
    resultat = dataframe.copy()

    resultat["datetime"] = pd.to_datetime(
        resultat["datetime"],
        errors="coerce",
    )

    for colonne in ["duration_seconds", "latitude", "longitude"]:
        resultat[colonne] = pd.to_numeric(
            resultat[colonne],
            errors="coerce",
        )

    resultat["observation_year"] = resultat["datetime"].dt.year
    resultat["observation_month"] = resultat["datetime"].dt.month
    resultat["observation_hour"] = resultat["datetime"].dt.hour

    resultat["text_features"] = (
        "city " + resultat["city"].fillna("<MANQUANT>").astype(str)
        + " state " + resultat["state"].fillna("<MANQUANT>").astype(str)
        + " country " + resultat["country"].fillna("<MANQUANT>").astype(str)
        + " shape " + resultat["shape"].fillna("<MANQUANT>").astype(str)
    )

    return resultat


## 9. Création du pipeline complet

`FunctionTransformer` applique la fonction de préparation à chaque jeu. Le vocabulaire TF-IDF et les médianes sont appris uniquement lors de `fit(X_train, y_train)`.

In [ ]:
FEATURES_NUMERIQUES = [
    "duration_seconds",
    "latitude",
    "longitude",
    "observation_year",
    "observation_month",
    "observation_hour",
]

preprocessing = ColumnTransformer(
    transformers=[
        (
            "texte",
            TfidfVectorizer(
                lowercase=True,
                min_df=2,
                max_features=10_000,
                ngram_range=(1, 2),
            ),
            "text_features",
        ),
        (
            "numerique",
            SimpleImputer(strategy="median"),
            FEATURES_NUMERIQUES,
        ),
    ]
)

modele_final = Pipeline(
    steps=[
        (
            "preparation_ligne",
            FunctionTransformer(preparer_releves, validate=False),
        ),
        ("preprocessing", preprocessing),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

modele_final


## 10. Entraînement de la chaîne complète

In [ ]:
modele_final.fit(X_train, y_train)
print("Entraînement de la chaîne complète terminé.")


## 11. Évaluation sur le test temporel

In [ ]:
y_pred = modele_final.predict(X_test)

precision_phase10 = precision_score(y_test, y_pred, zero_division=0)
recall_phase10 = recall_score(y_test, y_pred, zero_division=0)
accuracy_phase10 = accuracy_score(y_test, y_pred)

print(f"Precision : {precision_phase10:.2%}")
print(f"Recall : {recall_phase10:.2%}")
print(f"Accuracy : {accuracy_phase10:.2%}")


## 12. Matrice de confusion

In [ ]:
matrice_phase10 = confusion_matrix(y_test, y_pred)

df_matrice_phase10 = pd.DataFrame(
    matrice_phase10,
    index=["Réel : non-canular", "Réel : canular"],
    columns=["Prédit : non-canular", "Prédit : canular"],
)

df_matrice_phase10


## 13. Démonstration avec un relevé inventé

Un seul relevé brut entre dans le pipeline. La préparation, l'imputation, la vectorisation et la prédiction sont appliquées dans un seul appel.

In [ ]:
nouveau_releve = pd.DataFrame(
    [
        {
            "datetime": "2015-01-15 22:30:00",
            "city": "lille",
            "state": pd.NA,
            "country": "fr",
            "shape": "light",
            "duration_seconds": 120,
            "latitude": 50.6292,
            "longitude": 3.0573,
        }
    ]
)

prediction_nouveau_releve = modele_final.predict(nouveau_releve)[0]
probabilite_nouveau_releve = modele_final.predict_proba(nouveau_releve)[0, 1]

print("Relevé inventé :")
display(nouveau_releve)

if prediction_nouveau_releve == 1:
    print("Prédiction : canular")
else:
    print("Prédiction : non-canular")

print(
    f"Probabilité estimée de canular : {probabilite_nouveau_releve:.2%}"
)


## 14. Export des résultats

In [ ]:
resume_repartition.to_csv(
    PHASE10_DIR / "repartition_train_test_phase10.csv",
    index=False,
)

df_matrice_phase10.to_csv(
    PHASE10_DIR / "matrice_confusion_phase10.csv",
    index=True,
)

resultats_phase10 = pd.DataFrame(
    [
        {
            "modele": "Pipeline complet sans fuite",
            "date_coupure": date_coupure,
            "precision": precision_phase10,
            "recall": recall_phase10,
            "accuracy": accuracy_phase10,
            "nombre_train": len(df_train),
            "nombre_test": len(df_test),
            "prediction_releve_invente": int(prediction_nouveau_releve),
            "probabilite_canular_releve_invente": probabilite_nouveau_releve,
        }
    ]
)

resultats_phase10.to_csv(
    PHASE10_DIR / "resultats_modele_phase10.csv",
    index=False,
)

nouveau_releve.assign(
    prediction=prediction_nouveau_releve,
    probabilite_canular=probabilite_nouveau_releve,
).to_csv(
    PHASE10_DIR / "demonstration_releve_invente.csv",
    index=False,
)

print("Fichiers exportés dans :")
print(PHASE10_DIR)
